In [1]:
import os
os.chdir("..")       
print(os.getcwd())    

c:\college_project_Hostel_mess


In [2]:
import os
import numpy as np
import pandas as pd


In [3]:
os.makedirs("dataset", exist_ok=True)
os.makedirs("img", exist_ok=True)

In [4]:
RNG = np.random.default_rng(2026)

In [5]:
HOSTEL = 400
meals= ['Breakfast','Lunch','Dinner']
per_head ={"Breakfast":0.20,"Lunch":0.45,"Dinner":0.40}
turnout = {"Breakfast":0.55,"Lunch":0.80,"Dinner":0.85}

dates = pd.date_range("2026-01-01",periods=180)

In [6]:
df= pd.DataFrame({"date":np.repeat(dates,3),
                "meal":np.tile(meals,len(dates))})

n = len(df)

df

,date,meal
0,2026-01-01,Breakfast
1,2026-01-01,Lunch
2,2026-01-01,Dinner
3,2026-01-02,Breakfast
4,2026-01-02,Lunch
...,...,...
535,2026-06-28,Lunch
536,2026-06-28,Dinner
537,2026-06-29,Breakfast
538,2026-06-29,Lunch


In [7]:
df['day']= df.date.dt.day_name()
df['is_weekend'] = (df.date.dt.dayofweek >= 5).astype(int)
week = (df.date - dates[0]).dt.days //7
df["is_exam_week"]= week.isin([10,11,20,21]).astype(int)

In [8]:
base = df.meal.map(turnout) * HOSTEL
present = base - 35 * df.is_weekend - 30 *df.is_exam_week +RNG.normal(0,15,n)
df["students_present"] = np.clip(present, 100, HOSTEL).round().astype(int)

In [9]:
df["mess_rating"] = np.clip(RNG.normal(3.4,0.7,n),1,5).round(1)


In [10]:
ph = df.meal.map(per_head)
df["food_cooked_kg"] = (HOSTEL * ph * RNG.normal(1.0,0.03,n)).round(1)

In [11]:
appetite = 0.75 +0.06 * (df.mess_rating - 3)
eaten = df.students_present* ph* appetite
df['food_wasted_kg']= np.clip(df.food_cooked_kg - eaten,0,None).round(1)
df['waste_pct'] = (df.food_wasted_kg /df.food_cooked_kg*100).round(1)

In [12]:
df['food_wasted_kg']

0      47.8
1      75.4
2      72.2
3      43.0
4      52.1
       ... 
535    86.4
536    66.5
537    48.2
538    71.6
539    58.0
Name: food_wasted_kg, Length: 540, dtype: float64

In [13]:
df.to_csv("dataset/mess.csv", index=False)
print(f"{len(df)} rows saved to data/mess.csv")

540 rows saved to data/mess.csv


In [14]:
print(df.head())

        date       meal       day  is_weekend  is_exam_week  students_present  \
0 2026-01-01  Breakfast  Thursday           0             0               208   
1 2026-01-01      Lunch  Thursday           0             0               324   
2 2026-01-01     Dinner  Thursday           0             0               312   
3 2026-01-02  Breakfast    Friday           0             0               241   
4 2026-01-02      Lunch    Friday           0             0               330   

   mess_rating  food_cooked_kg  food_wasted_kg  waste_pct  
0          3.1            79.2            47.8       60.4  
1          2.5           180.4            75.4       41.8  
2          2.7           163.6            72.2       44.1  
3          4.0            82.0            43.0       52.4  
4          4.6           177.7            52.1       29.3  


In [15]:
print(df.shape)

(540, 10)
